In [4]:
import struct

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import torch.onnx

## Custom Dataset: MNISTDataset Abstraction Layer
**DataSet Link:** [MNIST Dataset](https://www.kaggle.com/datasets/hojjatk/mnist-dataset?resource=download)

In [5]:
class MNISTDataset(Dataset):
    def __init__(self, image_path, label_path):
        self.images = self.load_images(image_path)
        self.labels = self.load_labels(label_path)
        
    def __getitem__(self, idx):
        image = self.images[idx].float() / 255.0
        image = image.unsqueeze(0)
        image = F.pad(image, (2,2,2,2), value=0.0)
        label = self.labels[idx].long()
        return image, label
        
    def load_images(self, path):
        with open(path, "rb") as f:
            magic, n, rows, cols = struct.unpack('>IIII', f.read(16))
            data = torch.frombuffer(f.read(), dtype=torch.uint8)
        return data.reshape(n, rows, cols) # (60000, 28, 28)
    
    def load_labels(self, path):
        with open(path, 'rb') as f:
            magic, n = struct.unpack('>II', f.read(8))
            data = torch.frombuffer(f.read(), dtype=torch.uint8)
        return data # (60000)

    def __len__(self):
        return len(self.images)
        

In [6]:
train_ds = MNISTDataset(image_path='./mnist_dataset/train-images.idx3-ubyte', label_path='./mnist_dataset/train-labels.idx1-ubyte')
test_ds = MNISTDataset(image_path='./mnist_dataset/t10k-images.idx3-ubyte', label_path='./mnist_dataset/t10k-labels.idx1-ubyte')

train_loader = DataLoader(dataset=train_ds, shuffle=True, batch_size=32, drop_last=True)
test_loader = DataLoader(dataset=test_ds, batch_size=64, drop_last=True)

/tmp/ipykernel_58104/3657688393.py:16: UserWarning: The given buffer is not writable, and PyTorch does not support non-writable tensors. This means you can write to the underlying (supposedly non-writable) buffer using the tensor. You may want to copy the buffer to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /build/python-pytorch/src/pytorch/torch/csrc/utils/tensor_new.cpp:1581.)
  data = torch.frombuffer(f.read(), dtype=torch.uint8)


## CNN Custom Neural Net | Replica of LeNet5

In [7]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5), # (1, 32, 32) -> (6, 28, 28)
            nn.Tanh(), # tanh activation
            nn.AvgPool2d(kernel_size=2, stride=2), # (6, 28, 28) -> (6, 14, 14)
            
            nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5), # (6, 14, 14) -> (16, 10, 10)
            nn.Tanh(), # tanh activation
            nn.AvgPool2d(kernel_size=2, stride=2), # (16, 10, 10) -> (16, 5, 5)
            
            nn.Conv2d(in_channels=16, out_channels=120, kernel_size=5), # (16, 5, 5) -> (120, 1, 1)
            nn.Tanh(), # tanh activation
        )
        self.classifier = nn.Sequential(
            nn.Linear(120, 84), # 120 -> 84 hidden nodes
            nn.Tanh(), # tanh activation
            nn.Linear(84, 10), # 84 -> 10 output nodes
        )
        
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1) # Flatten the 1x1 grid to 1 Linear Tensor
        x = self.classifier(x)
        return x

## Training Phase

In [5]:
torch.manual_seed(123)
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Training
max_epoch = 10
model.train()
for epoch in range(max_epoch):
    for idx, (images, labels) in enumerate(train_loader):
        predictions = model(images)
        loss = criterion(predictions, labels)
        print(f"Batch Idx: {idx} | Epoch: {epoch+1} | Loss: {loss}")

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
print(model)

Batch Idx: 0 | Epoch: 1 | Loss: 2.278038501739502
Batch Idx: 1 | Epoch: 1 | Loss: 2.3165855407714844
Batch Idx: 2 | Epoch: 1 | Loss: 2.288052558898926
Batch Idx: 3 | Epoch: 1 | Loss: 2.302375078201294
Batch Idx: 4 | Epoch: 1 | Loss: 2.3170816898345947
Batch Idx: 5 | Epoch: 1 | Loss: 2.2943530082702637
Batch Idx: 6 | Epoch: 1 | Loss: 2.3132429122924805
Batch Idx: 7 | Epoch: 1 | Loss: 2.2952873706817627
Batch Idx: 8 | Epoch: 1 | Loss: 2.2816617488861084
Batch Idx: 9 | Epoch: 1 | Loss: 2.2828643321990967
Batch Idx: 10 | Epoch: 1 | Loss: 2.300122022628784
Batch Idx: 11 | Epoch: 1 | Loss: 2.322129487991333
Batch Idx: 12 | Epoch: 1 | Loss: 2.292613983154297
Batch Idx: 13 | Epoch: 1 | Loss: 2.2978901863098145
Batch Idx: 14 | Epoch: 1 | Loss: 2.2872159481048584
Batch Idx: 15 | Epoch: 1 | Loss: 2.294457197189331
Batch Idx: 16 | Epoch: 1 | Loss: 2.2941198348999023
Batch Idx: 17 | Epoch: 1 | Loss: 2.2803423404693604
Batch Idx: 18 | Epoch: 1 | Loss: 2.287560224533081
Batch Idx: 19 | Epoch: 1 | Los

KeyboardInterrupt: 

## Testing Phase

In [74]:
# Evaluation
model.eval()

correct = 0
total = correct

with torch.no_grad():
    for idx, (images, labels) in enumerate(test_loader):
        predictions = model(images)
        loss = criterion(predictions, labels)
        predicted_classes = predictions.argmax(dim=1)
        correct += (predicted_classes == labels).sum().item()
        total += labels.size(0)
    accuracy = correct / total
    print(f"Accuracy: {accuracy} | Correct: {correct} / Total: {total}")

        

Accuracy: 0.9886818910256411 | Correct: 9871 / Total: 9984


## Example of model saving, loading and exporting to onnx format

In [75]:
torch.save(model.state_dict(), "lenet5.pth")

In [8]:
xmodel = CNN()
xmodel.load_state_dict(torch.load("lenet5.pth"))

<All keys matched successfully>

In [9]:
dummy_input = torch.zeros(1, 1, 32, 32)
torch.onnx.export(
    xmodel,
    dummy_input,
    'lenet5.onnx',
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}},
)

/tmp/ipykernel_58104/3926067502.py:2: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  torch.onnx.export(
/tmp/ipykernel_58104/3926067502.py:2: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `CNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `CNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 1 of general pattern rewrite rules.


/usr/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 20},
            producer_name='pytorch',
            producer_version='2.10.0',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[batch_size,1,32,32]>
            ),
            outputs=(
                %"output"<FLOAT,[batch_size,10]>
            ),
            initializers=(
                %"features.0.weight"<FLOAT,[6,1,5,5]>{TorchTensor(...)},
                %"features.0.bias"<FLOAT,[6]>{TorchTensor<FLOAT,[6]>(Parameter containing: tensor([ 0.1739, -0.2825,  0.1049, -0.0348, -0.2123,  0.2169], requires_grad=True), name='features.0.bias')},
                %"features.3.weight"<FLOAT,[16,6,5,5]>{TorchTensor(...)},
                %"features.3.bias"<FLOAT,[16]>{TorchTensor(...)},
                %"features.6.weight"<FLOAT,[120,16,5,5]>{TorchTensor(...)},
                %"featur